# TP2 — CNN and Transfer Learning: Age Estimation from Face Images

---

## Objectives

By the end of this session, you will be able to:
1. Build and train a **convolutional neural network from scratch** on a real (not toy) regression task
2. Apply **transfer learning** with an ImageNet-pretrained CNN, using both the *feature extraction* and *fine-tuning* strategies
3. Design a controlled comparison to argue **when and why transfer learning helps**
4. Read a trained CNN's internal representations through **feature map visualisation**

## Prerequisites (CM3)
- CNN basics — convolution, padding, stride, pooling (from ML course, not repeated here)
- Architecture evolution — LeNet → AlexNet → VGG → **ResNet** (residual connections)
- Transfer learning — feature extraction vs. fine-tuning, why pretrained features transfer

## The task

You will train models to predict a person's **age from a face photo** — a regression problem
(continuous output), unlike TP1's classification task. The dataset is a curated subset of
**APPA-REAL** (Agustsson et al., FG 2017), a public face-age-estimation benchmark.

## Deliverables
**Send your GitHub repository link to [xchang3@slb.com](mailto:xchang3@slb.com) and submit on GitHub before the deadline.**
```
your-repo/
├── tp2-cnn-transfer-learning.ipynb   ← all cells executed top-to-bottom, outputs visible
└── report.pdf                        ← 1–2 pages, maximum 2 pages
```

**Deadline**: **23:59, October 19th**

Your report should answer the following sections:
1. **Part 1 — A CNN from Scratch**: answer the two questions at the end of Part 1
2. **Experiment 1 — Comparing the Three Approaches**: fill in the Results and Analysis section
3. **Experiment 2 — Does Data Augmentation Help Fine-tuning?**: fill in the Results and Analysis section
4. **Experiment 3 — How Much of the Network Should You Fine-tune?**: fill in the Results and Analysis section
5. **Overall Comparison**: answer the final report questions
6. **Experiment 4 — Feature Map Visualisation**: answer the Results and Analysis question

Results must be discussed analytically — cite specific numbers, reference figures by their
titles, and connect observations to theory from CM3.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import os, csv, zipfile, copy

# ── Device setup ──────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'PyTorch version: {torch.__version__}')

torch.manual_seed(0)
NUM_WORKERS = 2 if device.type == 'cuda' else 0
IMG_SIZE = 128   # keep training fast on a free Colab GPU / CPU

---

## 1. Dataset: APPA-REAL (age subset)

| Property | Value |
|----------|-------|
| Input | RGB face crop |
| Target | `age` (years, continuous — a **regression** task) |
| Train / Valid / Test | ~4000 / ~1200 / ~1200 images |
| Image size (as shipped) | 160×160, resized to `IMG_SIZE`×`IMG_SIZE` in the pipeline |

Unlike Fashion-MNIST in TP1, this is a **real-world dataset**. The label is `real_age` — the
person's true, documented age (not `apparent_age`, a noisier "how old do they look" estimate
averaged over several human raters, which this TP doesn't use). Coverage is imbalanced:
the train split mirrors the original dataset's skew toward 20–40 year-olds, and even after
stratified sampling, images of people over 60 remain scarce in every split simply because the
source dataset has few of them — check the histogram below once you've loaded the data.

**Download**: the dataset is hosted on Google Drive and pulled with `gdown` below (plain
`urllib` downloads fail on Drive files this size). 

In [ ]:
DATA_FILE_ID = "18Ipvzj-gMH_q57FgaKMl_IMBoGFLP34i"   # appa_real_subset.zip on Google Drive
DATA_DIR = "./appa_real_subset"

if not os.path.exists(DATA_DIR):
    import gdown   # preinstalled on Colab; `pip install gdown` if running locally
    zip_path = "appa_real_subset.zip"
    print("Downloading dataset...")
    gdown.download(id=DATA_FILE_ID, output=zip_path, quiet=False)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(DATA_DIR)
    os.remove(zip_path)
    print("Done.")
else:
    print("Dataset already present.")

### 1.1 — Dataset and DataLoader

Complete the `# TODO` sections below: in `AgeDataset.__getitem__`, load the image, apply the
transform, and return `(image_tensor, age)` as a `(float32 tensor, float32 tensor)` pair. Then,
after `train_set`/`valid_set`/`test_set` are built, wrap each in a `DataLoader`.

In [ ]:
class AgeDataset(Dataset):
    def __init__(self, root, split, transform):
        self.root = os.path.join(root, split)
        self.transform = transform
        labels_path = os.path.join(root, f"{split}_labels.csv")
        with open(labels_path) as f:
            reader = csv.DictReader(f)
            self.samples = [(row["file_name"], float(row["age"])) for row in reader]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_name, age = self.samples[idx]
        # TODO: open the image at os.path.join(self.root, file_name) with PIL, convert to RGB
        img = None

        # TODO: apply self.transform to img
        img = None

        # TODO: return (img, age) — wrap age in torch.tensor(age, dtype=torch.float32)
        return None

# ImageNet stats: required for the pretrained ResNet in Part 2, harmless for the
# from-scratch CNN in Part 1 (it just standardises the inputs either way).
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

train_set = AgeDataset(DATA_DIR, "train", transform)
valid_set = AgeDataset(DATA_DIR, "valid", transform)
test_set  = AgeDataset(DATA_DIR, "test",  transform)

# TODO: create train_loader, valid_loader, test_loader with DataLoader(...)
#       — batch_size=32, num_workers=NUM_WORKERS for all three; shuffle=True for
#       train_loader only (valid/test order shouldn't matter, and shuffling them
#       would just waste time)
train_loader = None
valid_loader = None
test_loader = None

print(f"Train: {len(train_set)} | Valid: {len(valid_set)} | Test: {len(test_set)}")

### 1.2 — Explore the data

Before building anything, look at what you're modelling: sample images and the age distribution
per split. Run the cell below (nothing to fill in).


In [ ]:
def denormalize(img_tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (img_tensor * std + mean).clamp(0, 1)

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for i, ax in enumerate(axes):
    img, age = train_set[i * 150]
    ax.imshow(denormalize(img).permute(1, 2, 0).numpy())
    ax.set_title(f"age={age:.0f}")
    ax.axis("off")
plt.suptitle("Sample training images")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 3))
for split_set, label in [(train_set, "train"), (valid_set, "valid"), (test_set, "test")]:
    ages = [a for _, a in split_set.samples]
    ax.hist(ages, bins=18, alpha=0.5, label=label)
ax.set_xlabel("age (years)"); ax.set_ylabel("count"); ax.legend()
ax.set_title("Age distribution per split")
plt.tight_layout()
plt.show()

---

## Part 1 — A CNN from scratch

### 1.3 — Model Architecture

Architecture (all convolutions use `kernel_size=3, padding=1`, so spatial size only changes at
the pooling layers):

```
Input (3×128×128)
 → Conv(3→32)  + BatchNorm + ReLU + MaxPool(2)   → 32×64×64
 → Conv(32→64) + BatchNorm + ReLU + MaxPool(2)   → 64×32×32
 → Conv(64→128)+ BatchNorm + ReLU + MaxPool(2)   → 128×16×16
 → Conv(128→256)+BatchNorm + ReLU + AdaptiveAvgPool(1) → 256×1×1
 → Flatten → Linear(256→64) → ReLU → Linear(64→1)   [age prediction]
```

Translate this diagram into `self.features` and `self.regressor` below. You've built an
analogous stack of `Linear → BatchNorm → ReLU` blocks in TP1 — the 2D counterparts
(`nn.Conv2d`, `nn.BatchNorm2d`) follow the same logic. Check the PyTorch docs for their exact
signatures if needed.


In [ ]:
class ScratchCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 3 -> 32
            ???,  # TODO: convolution
            ???,  # TODO: batch norm
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 2: 32 -> 64
            ???,  # TODO: convolution
            ???,  # TODO: batch norm
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 3: 64 -> 128
            ???,  # TODO: convolution
            ???,  # TODO: batch norm
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 4: 128 -> 256, then global average pool to 1x1
            ???,  # TODO: convolution
            ???,  # TODO: batch norm
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.regressor = nn.Sequential(
            nn.Flatten(),
            ???,  # TODO: first linear layer
            nn.ReLU(inplace=True),
            ???,  # TODO: output linear layer -> single value
        )

    def forward(self, x):
        x = self.features(x)
        return self.regressor(x)   # shape (batch, 1) — reshaped to (batch,) in train/eval below

scratch_model = ScratchCNN().to(device)
n_params = sum(p.numel() for p in scratch_model.parameters())
print(f"ScratchCNN parameters: {n_params:,}")

# Sanity check
with torch.no_grad():
    dummy = torch.zeros(4, 3, IMG_SIZE, IMG_SIZE).to(device)
    out = scratch_model(dummy)
    print(f"Output shape: {out.shape}")  # Expected: torch.Size([4, 1])

### 1.4 — Loss Function and Optimiser

For regression, the standard training loss is **Mean Squared Error** (penalises large errors
more heavily — recall CM1's discussion of MSE vs. cross-entropy). But MSE is in *squared years*,
which is hard to interpret. So while we **train** with MSE, we **report** Mean Absolute Error
(MAE) — "the model is off by X years on average" — as the evaluation metric everyone actually
cares about.


In [ ]:
# Q1: Which loss function is appropriate for a continuous-valued target?
#     Options: nn.MSELoss()  or  nn.CrossEntropyLoss()
criterion = None  # TODO

# Q2: Optimizer — same default choice as TP1, lr=1e-3
optimizer = None  # TODO

if criterion is not None and optimizer is not None:
    print(f"Loss: {type(criterion).__name__}, Optimizer: {type(optimizer).__name__}")

### 1.5 — Training Loop

You wrote this loop once already in TP1. Adapt it here for regression: complete the `# TODO`
blocks in `train_epoch` and `evaluate` below. Two things differ from TP1's classification loop:
- The model's raw output has shape `(batch, 1)`; `ages` has shape `(batch,)` — reshape
  predictions with `.view(-1)` before computing the loss, or PyTorch will silently broadcast
  the mismatched shapes into something meaningless.
- There's no "accuracy" for a continuous target — `evaluate` should also accumulate the sum of
  absolute errors, so it can return MAE (in years) alongside the loss.

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    """One epoch of training. Returns average MSE loss."""
    model.train()
    total_loss, total = 0.0, 0
    for images, ages in loader:
        images, ages = images.to(device), ages.to(device)
        # TODO: zero the gradients, forward pass (reshape output to (batch,) with .view(-1)),
        #       compute the loss, backpropagate, and take an optimizer step
        loss = None

        total_loss += loss.item() * images.size(0)
        total += images.size(0)
    return total_loss / total

def evaluate(model, loader, criterion):
    """Returns (avg MSE loss, MAE in years)."""
    model.eval()
    total_loss, total_abs_err, total = 0.0, 0.0, 0
    with torch.no_grad():
        for images, ages in loader:
            images, ages = images.to(device), ages.to(device)
            # TODO: forward pass (reshape to (batch,)), compute the loss,
            #       and accumulate both total_loss and total_abs_err
            pass

            total += images.size(0)
    return total_loss / total, total_abs_err / total

def plot_history(history, title="Training Curves"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    epochs = range(1, len(history["train_loss"]) + 1)
    ax1.plot(epochs, history["train_loss"], label="Train")
    ax1.plot(epochs, history["val_loss"], label="Val", linestyle="--")
    ax1.set_title("MSE Loss"); ax1.set_xlabel("Epoch"); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(epochs, history["train_mae"], label="Train")
    ax2.plot(epochs, history["val_mae"], label="Val", linestyle="--")
    ax2.set_title("MAE (years)"); ax2.set_xlabel("Epoch"); ax2.legend(); ax2.grid(alpha=0.3)
    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

def run_training(model, loader, criterion, optimizer, num_epochs, label=""):
    """Trains for num_epochs, then restores the weights from the epoch with the best
    (lowest) validation MAE — not necessarily the last epoch. Without this, a model that
    starts overfitting partway through would be judged on its worst, most-overfit state
    just because training happened to stop there."""
    history = {"train_loss": [], "val_loss": [], "train_mae": [], "val_mae": []}
    best_val_mae, best_state, best_epoch = float("inf"), None, 0

    if label:
        print(label)
    print("Epoch | Train Loss | Val Loss | Train MAE | Val MAE")
    print("-" * 54)
    for epoch in range(1, num_epochs + 1):
        tr_loss = train_epoch(model, loader, criterion, optimizer)
        _, tr_mae = evaluate(model, loader, criterion)
        va_loss, va_mae = evaluate(model, valid_loader, criterion)
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_mae"].append(tr_mae)
        history["val_mae"].append(va_mae)
        print(f"{epoch:>5} | {tr_loss:>10.2f} | {va_loss:>8.2f} | {tr_mae:>9.2f} | {va_mae:>7.2f}")

        if va_mae < best_val_mae:
            best_val_mae, best_epoch = va_mae, epoch
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    print(f"\n→ Restored weights from epoch {best_epoch} (best val MAE = {best_val_mae:.2f})")
    return history

In [ ]:
# ── Train the from-scratch CNN ─────────────────────────────────────────────────
NUM_EPOCHS = 20
scratch_history = run_training(scratch_model, train_loader, criterion, optimizer, NUM_EPOCHS)

_, scratch_test_mae = evaluate(scratch_model, test_loader, criterion)
print(f"\nFrom-scratch CNN — Test MAE: {scratch_test_mae:.2f} years")
plot_history(scratch_history, title="From-scratch CNN")

**Section 1 of your report** is based on what you just built and observed:
- What test MAE did you get? Is it good enough to be useful (compare to: a model that always
  predicts the average age of the training set)?
- Look at the train/val MAE gap in the plot — is the model overfitting, underfitting, or roughly
  balanced with ~4000 training images?

---

## Part 2 — Transfer Learning

In this section, instead of learning convolutional filters
from scratch, we start from a network already trained on ImageNet. This lets us directly
compare transfer learning against the from-scratch baseline you just built.

### 2.1 — Load a pretrained ResNet and adapt it for regression

We use `resnet18` pretrained on ImageNet. Its final layer (`model.fc`) was trained for
1000-class classification — you need to replace it with a single-output regression head.

Complete `build_resnet_regressor()` below:
1. Load a ResNet-18 with ImageNet-pretrained weights (check `torchvision.models` — the current
   API takes a `weights=` argument)
2. Find the input size of its existing classification head — you don't need to know it in
   advance, read it directly off the loaded model
3. Replace that head with a layer producing a single output value

In [ ]:
def build_resnet_regressor():
    # TODO: load resnet18 pretrained on ImageNet
    model = None

    # TODO: read off the input size of the original classification head
    in_features = None

    # TODO: replace the classification head with a single-output regression layer

    return model

resnet_model = build_resnet_regressor().to(device)
print(resnet_model.fc)

### 2.2 — Stage 1: Feature Extraction (frozen backbone)

Freeze every parameter *except* the new `fc` head — the pretrained convolutional filters are
used as a fixed feature extractor, and only the regression head is trained. This is fast and
works well when your dataset is small and similar in domain to ImageNet (natural images —
faces are a reasonable, if imperfect, match).

Complete `freeze_backbone`: only the `fc` layer's parameters should remain trainable. Check the
`frozen_trainable / frozen_total` parameter counts printed below — does the ratio match what
you'd expect from a ResNet-18's total parameter count?


In [ ]:
def freeze_backbone(model):
    for name, param in model.named_parameters():
        # TODO: disable gradient tracking for every parameter that is not part of `fc`
        pass

freeze_backbone(resnet_model)
frozen_trainable = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
frozen_total = sum(p.numel() for p in resnet_model.parameters())
print(f"Trainable parameters: {frozen_trainable:,} / {frozen_total:,}")

criterion = nn.MSELoss()
# Note: only pass fc.parameters() to the optimizer — no point tracking frozen params
optimizer = optim.Adam(resnet_model.fc.parameters(), lr=1e-3)

NUM_EPOCHS_STAGE1 = 20
frozen_history = run_training(
    resnet_model, train_loader, criterion, optimizer, NUM_EPOCHS_STAGE1,
    label="\nStage 1 — Feature extraction (frozen backbone)",
)

_, frozen_test_mae = evaluate(resnet_model, test_loader, criterion)
print(f"\nFrozen-backbone ResNet — Test MAE: {frozen_test_mae:.2f} years")

### 2.3 — Stage 2: Fine-tuning (unfreeze the backbone)

Now unfreeze the whole network and continue training — but at a **much lower learning rate**
than Stage 1. The backbone's ImageNet features are already good; a large learning rate here
would destroy them (catastrophic forgetting) before the head has a chance to guide useful
updates.

Complete `unfreeze_all`, then create a **new** optimizer over every one of `resnet_model`'s
parameters. Pick a learning rate an order of magnitude below Stage 1's — and be ready to justify
in your report why a smaller rate matters here specifically.


In [ ]:
def unfreeze_all(model):
    for param in model.parameters():
        # TODO: re-enable gradient tracking
        pass

unfreeze_all(resnet_model)

# TODO: new optimizer over resnet_model.parameters(), lr ~10x smaller than Stage 1
optimizer = None

NUM_EPOCHS_STAGE2 = 20
finetune_history = run_training(
    resnet_model, train_loader, criterion, optimizer, NUM_EPOCHS_STAGE2,
    label="Stage 2 — Fine-tuning (full network, low LR)",
)

_, finetune_test_mae = evaluate(resnet_model, test_loader, criterion)
print(f"\nFine-tuned ResNet — Test MAE: {finetune_test_mae:.2f} years")

---

## Experiment 1 — Comparing the Three Approaches

**Objective**: Quantify the effect of transfer learning, and of *how much* of the network you
fine-tune, on final performance.

In [ ]:
results = {
    "From-scratch CNN": scratch_test_mae,
    "ResNet (frozen)": frozen_test_mae,
    "ResNet (fine-tuned)": finetune_test_mae,
}

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(results.keys(), results.values(), color=["#888", "#4C72B0", "#DD8452"])
ax.set_ylabel("Test MAE (years, lower is better)")
ax.set_title("From-scratch vs. Transfer Learning")
for bar, val in zip(bars, results.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, val, f"{val:.2f}", ha="center", va="bottom")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

for name, mae in results.items():
    print(f"{name:>22}: {mae:.2f} years MAE")

**Results and Analysis** *(complete after running)*
- Rank the three models. Does the ranking match what CM3 predicted about transfer learning?
- Compare Stage 1 (frozen) vs. Stage 2 (fine-tuned) — did unfreezing help, hurt, or barely
  change things? Given only ~4000 training images, what does that tell you about the risk of
  fine-tuning a large pretrained network on a small dataset?
- The pretrained ResNet saw *object categories*, not faces, during pretraining. Why would its
  features transfer to a face-age-estimation task at all?

---

## Experiment 2 — Does Data Augmentation Help Fine-tuning?

Stage 2's fine-tuning run shows a warning sign for overfitting: training loss keeps dropping
long after validation loss has flattened out, opening up a large train/val gap. With only
~4000 training images, **data augmentation** — randomly perturbing each training image (flip,
rotation, color jitter) so the model sees a different version every epoch — is the standard
fix. We repeat the two-stage transfer-learning procedure with an augmented training set and
compare.

Complete `augment_transform` below: same `Resize` / `ToTensor` / `Normalize` as the original
`transform`, plus a random horizontal flip, a small random rotation, and color jitter.

In [ ]:
# TODO: build augment_transform — same Resize/ToTensor/Normalize as `transform` above,
#       plus transforms.RandomHorizontalFlip(), transforms.RandomRotation(10), and
#       transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)
augment_transform = None

train_set_aug = AgeDataset(DATA_DIR, "train", augment_transform)
train_loader_aug = DataLoader(train_set_aug, batch_size=32, shuffle=True, num_workers=NUM_WORKERS)

resnet_aug = build_resnet_regressor().to(device)
criterion = nn.MSELoss()

# Stage 1: same warm-up as before, frozen backbone — augmentation barely matters here since
# only the 513-parameter head is trained. Uses run_training (not a bare loop) so this Stage 1
# gets the same best-epoch checkpointing as every other run in this notebook — otherwise
# Stage 2 would start from a different quality of warm-up than the frozen/full-FT baselines,
# confounding the comparison.
freeze_backbone(resnet_aug)
optimizer = optim.Adam(resnet_aug.fc.parameters(), lr=1e-3)
run_training(
    resnet_aug, train_loader_aug, criterion, optimizer, NUM_EPOCHS_STAGE1,
    label="Stage 1 — Feature extraction (frozen backbone, augmented)",
)

# Stage 2: unfreeze everything and fine-tune on augmented data — this is the stage that showed
# overfitting (train loss << val loss) without augmentation
unfreeze_all(resnet_aug)
optimizer = optim.Adam(resnet_aug.parameters(), lr=1e-4)

aug_history = run_training(
    resnet_aug, train_loader_aug, criterion, optimizer, NUM_EPOCHS_STAGE2,
    label="Stage 2 — Fine-tuning with data augmentation",
)

_, aug_test_mae = evaluate(resnet_aug, test_loader, criterion)
print(f"\nFine-tuned ResNet + augmentation — Test MAE: {aug_test_mae:.2f} years")
plot_history(aug_history, title="Fine-tuned ResNet with Data Augmentation")

**Results and Analysis** *(complete after running)*
- Compare `aug_test_mae` to the earlier fine-tuned ResNet's `finetune_test_mae` (no
  augmentation). Did augmentation help, and by how much?
- Compare the train/val loss gap in the plot above to Stage 2's plot without augmentation
  (`finetune_history`, a few cells up). Did augmentation close the gap? Is that consistent with
  augmentation acting as a regulariser?
- Now compare the **best validation MAE** reached in each run to the **test MAE** difference
  you found in the first question. If the two runs' best val MAE end up close while their test
  MAE differs more noticeably, what does that tell you about relying on a ~1200-image
  validation set to judge whether a training change (like augmentation) actually helped? Is
  val MAE alone always a reliable signal here?
- Note: `train_mae` here is computed over `train_loader_aug`, so the random flips/rotations/
  color jitter are still applied when it's evaluated — it isn't directly comparable to
  `finetune_history`'s train MAE. Focus on the train-vs-val gap *within this experiment*, not
  the absolute train MAE value.

---

## Experiment 3 — How Much of the Network Should You Fine-tune?

Stage 2 unfroze the *entire* ResNet-18. But you don't have to go all-or-nothing: a common
middle ground is to unfreeze only the last one (or a few) residual blocks. Before picking one,
let's look at how ResNet-18's parameters are actually distributed across its stages.

In [ ]:
inspect_model = build_resnet_regressor()
print(f"{'Block':<16}{'Parameters':>14}")
print("-" * 30)
for name, module in inspect_model.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f"{name:<16}{n:>14,}")
    if name.startswith("layer"):
        for sub_name, sub_module in module.named_children():
            sub_n = sum(p.numel() for p in sub_module.parameters())
            print(f"  └─ {name}.{sub_name:<10}{sub_n:>10,}")

As the breakdown above shows, `layer4` alone accounts for most of the network's parameters,
and within it, `layer4.1` — the last residual block, closest to the classification head, whose
features are the most task-specific — is the single largest block. Unfreezing just `layer4.1`
(plus `fc`) while keeping everything earlier (`layer1`–`layer3` and `layer4.0`) frozen still
holds a sizeable chunk of ResNet-18's parameters (~40%), but it's a meaningfully smaller and
cheaper update than unfreezing the whole network.

Complete `freeze_all_but` below, then build an optimizer over only the parameters it left
trainable — unlike Stage 1, where `.fc.parameters()` already gave you exactly the right subset,
here you'll need to filter `resnet_partial.parameters()` yourself by `requires_grad`.

In [ ]:
def freeze_all_but(model, unfreeze_prefixes):
    for name, param in model.named_parameters():
        # TODO: set param.requires_grad = True if `name` starts with any prefix in
        #       `unfreeze_prefixes`, otherwise False
        pass

resnet_partial = build_resnet_regressor().to(device)
criterion = nn.MSELoss()

# Stage 1: identical warm-up to before — frozen backbone, train the head only. Uses
# run_training (not a bare loop) so this Stage 1 gets the same best-epoch checkpointing as
# every other run in this notebook — otherwise Stage 2 would start from a different quality
# of warm-up than the frozen/full-FT baselines, confounding the comparison.
freeze_backbone(resnet_partial)
optimizer = optim.Adam(resnet_partial.fc.parameters(), lr=1e-3)
run_training(
    resnet_partial, train_loader, criterion, optimizer, NUM_EPOCHS_STAGE1,
    label="Stage 1 — Feature extraction (frozen backbone, before partial fine-tune)",
)

# Stage 2: unfreeze only layer4.1 + fc, fine-tune at the same low LR as full fine-tuning
freeze_all_but(resnet_partial, ["layer4.1", "fc"])
partial_trainable = sum(p.numel() for p in resnet_partial.parameters() if p.requires_grad)
partial_total = sum(p.numel() for p in resnet_partial.parameters())
print(f"Trainable parameters (layer4.1 + fc only): {partial_trainable:,} / {partial_total:,}")

# TODO: new optimizer over only the parameters `freeze_all_but` left trainable —
#       filter resnet_partial.parameters() by .requires_grad, unlike Stage 1 where
#       .fc.parameters() already gave you exactly the right subset
optimizer = None

partial_history = run_training(
    resnet_partial, train_loader, criterion, optimizer, NUM_EPOCHS_STAGE2,
    label="\nStage 2 — Partial fine-tuning (layer4.1 + fc only)",
)

_, partial_test_mae = evaluate(resnet_partial, test_loader, criterion)
print(f"\nPartial fine-tune (layer4.1+fc) ResNet — Test MAE: {partial_test_mae:.2f} years")
plot_history(partial_history, title="Partial Fine-tuning (layer4.1 + fc only)")

**Results and Analysis** *(complete after running)*
- Where does partial fine-tuning (`partial_test_mae`) fall relative to frozen
  (`frozen_test_mae`) and full fine-tuning (`finetune_test_mae`)?
- Partial fine-tuning trains far fewer parameters than full fine-tuning (compare the counts
  printed above) but far more than the frozen head. Was the accuracy trade-off worth it here?
- If you only had, say, 500 training images instead of 4000, would you expect full fine-tuning
  to look relatively better or worse compared to partial fine-tuning? Why?

In [ ]:
all_results = {
    "From-scratch CNN": scratch_test_mae,
    "ResNet (frozen)": frozen_test_mae,
    "ResNet (partial FT)": partial_test_mae,
    "ResNet (full FT)": finetune_test_mae,
    "ResNet (full FT + aug)": aug_test_mae,
}

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(all_results.keys(), all_results.values(),
              color=["#888", "#4C72B0", "#55A868", "#DD8452", "#C44E52"])
ax.set_ylabel("Test MAE (years, lower is better)")
ax.set_title("All Five Approaches Compared")
for bar, val in zip(bars, all_results.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, val, f"{val:.2f}", ha="center", va="bottom")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

for name, mae in all_results.items():
    print(f"{name:>24}: {mae:.2f} years MAE")

In [ ]:
def best_epoch_stats(history):
    """Returns (1-indexed epoch, val MAE) of the epoch with the lowest val MAE."""
    best_idx = int(np.argmin(history["val_mae"]))
    return best_idx + 1, history["val_mae"][best_idx]

scratch_best_epoch, scratch_best_val_mae = best_epoch_stats(scratch_history)
frozen_best_epoch, frozen_best_val_mae = best_epoch_stats(frozen_history)
partial_best_epoch, partial_best_val_mae = best_epoch_stats(partial_history)
finetune_best_epoch, finetune_best_val_mae = best_epoch_stats(finetune_history)
aug_best_epoch, aug_best_val_mae = best_epoch_stats(aug_history)

summary = pd.DataFrame([
    {"Method": "From-scratch CNN",       "Epochs": len(scratch_history["val_mae"]),
     "Best epoch": scratch_best_epoch,   "Val MAE (best)": scratch_best_val_mae,
     "Test MAE": scratch_test_mae,       "Trainable params": n_params,
     "Trainable %": 100 * n_params / n_params},
    {"Method": "ResNet (frozen)",        "Epochs": len(frozen_history["val_mae"]),
     "Best epoch": frozen_best_epoch,    "Val MAE (best)": frozen_best_val_mae,
     "Test MAE": frozen_test_mae,        "Trainable params": frozen_trainable,
     "Trainable %": 100 * frozen_trainable / frozen_total},
    {"Method": "ResNet (partial FT)",    "Epochs": len(partial_history["val_mae"]),
     "Best epoch": partial_best_epoch,   "Val MAE (best)": partial_best_val_mae,
     "Test MAE": partial_test_mae,       "Trainable params": partial_trainable,
     "Trainable %": 100 * partial_trainable / partial_total},
    {"Method": "ResNet (full FT)",       "Epochs": len(finetune_history["val_mae"]),
     "Best epoch": finetune_best_epoch,  "Val MAE (best)": finetune_best_val_mae,
     "Test MAE": finetune_test_mae,      "Trainable params": frozen_total,
     "Trainable %": 100.0},
    {"Method": "ResNet (full FT + aug)", "Epochs": len(aug_history["val_mae"]),
     "Best epoch": aug_best_epoch,       "Val MAE (best)": aug_best_val_mae,
     "Test MAE": aug_test_mae,           "Trainable params": frozen_total,
     "Trainable %": 100.0},
])
summary.round(2)

**Overall comparison — final report questions**

The table above summarises all five approaches: how many epochs each was trained for, which
epoch actually had the best validation MAE, and the resulting val/test MAE and trainable
parameter count. Use it to answer:
- Rank all five approaches. Does augmentation give the biggest additional gain on top of full
  fine-tuning, or is the effect small compared to going from frozen → fine-tuned in the first
  place?
- Look at the **Best epoch** column relative to the 20 epochs each model was given. Which
  models used (close to) the full budget, and which ones peaked early and then plateaued or
  got worse? What does that tell you about how many epochs each strategy actually needs?
- Given the trainable-parameter counts, which approach looks like the best accuracy-per-
  training-cost trade-off for a dataset of this size?

---

## Experiment 4 — Feature Map Visualisation

What does the network actually "see"? We register a hook on an early convolutional layer of
the fine-tuned ResNet and visualise its activations for one sample image.

In [ ]:
sample_img, sample_age = valid_set[0]
activations = {}

def hook_fn(module, input, output):
    activations["layer1"] = output.detach().cpu()

handle = resnet_model.layer1.register_forward_hook(hook_fn)
resnet_model.eval()
with torch.no_grad():
    _ = resnet_model(sample_img.unsqueeze(0).to(device))
handle.remove()

fmap = activations["layer1"][0]   # (channels, H, W)
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(fmap[i].numpy(), cmap="viridis")
    ax.axis("off")
plt.suptitle(f"layer1 feature maps (first 16 of {fmap.shape[0]} channels) — true age {sample_age:.0f}")
plt.tight_layout()
plt.show()

**Results and Analysis** *(complete after running)*
- Do the different channels seem to respond to different things (edges, textures, regions of
  the face)? Pick two channels and describe what each seems to be detecting.